# Vector Search Retrieval Quality Diagnosis

### Source : KNOWLEDGE_RETRIEVAL_3/vector_search_retrieval_diagnosis.ipynb


### Maps to exam bullet:
- Given a Databricks agent with documented retrieval quality problems, diagnose which Vector
  Search configuration is the root cause, and select the remediation that most directly improves
  the signal quality of retrieved context.

### The idea

A real Vector Search endpoint and Delta Sync Index, on a shipping-policy corpus where the same zone has **two versions over time**. 
The fix is metadata filtering -- first with a hardcoded cutoff, then selected dynamically per query.


## Setup
A `.env` file with `DATABRICKS_HOST` and `DATABRICKS_TOKEN`

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.getenv("DATABRICKS_HOST"), "DATABRICKS_HOST not set -- check your .env file"
assert os.getenv("DATABRICKS_TOKEN"), "DATABRICKS_TOKEN not set -- check your .env file"

## Imports and config

In [ ]:
import numpy as np
import mlflow
from databricks_langchain import DatabricksEmbeddings

# Real Databricks Foundation Model API endpoint -- this call hits your workspace.
EMBEDDING_ENDPOINT = "databricks-gte-large-en"
embeddings = DatabricksEmbeddings(endpoint=EMBEDDING_ENDPOINT)

# TODO: change "oliver@mlops-media.com" to your own workspace user email
mlflow.set_experiment("/Users/oliver@mlops-media.com/vector_search_retrieval_diagnosis")
print(f"Using embedding endpoint: {EMBEDDING_ENDPOINT}")

---
## 1 - A real Vector Search endpoint, with metadata filtering on dates

We create a Unity Catalog Delta table, a real Vector Search endpoint, and a Delta Sync Index, then
add an `effective_date` column. Each chunk is also enriched with semantic metadata before indexing
(document title + section)

This section creates real, billable Databricks resources (an endpoint + an index). A cleanup cell
is provided at the end -- run it when you're done experimenting.


In [ ]:
import sys
sys.path.append(os.path.abspath(".."))
from _utils.sql_utils import sql_call
from databricks.sdk import WorkspaceClient

DATABRICKS_CLIENT = WorkspaceClient()

VS_CATALOG, VS_SCHEMA = "demo", "context"
SOURCE_TABLE = f"{VS_CATALOG}.{VS_SCHEMA}.shipping_policy_chunks"
WAREHOUSE_ID = os.getenv("SQL_WAREHOUSE_ID")

DOC_TITLE = "Shipping Policy by Zone"


In [ ]:
# Same shipping-policy domain as before, but now with a real effective_date per chunk --
# and Zone B deliberately has TWO versions: a 2024 rate and a 2026 rate increase.
raw_chunks = [
    {"chunk_id": "zone_a_2024", "zone": "A", "effective_date": "2024-01-01", "is_current": True, "section": "Zone A rates", "text":
        "Zone A covers domestic shipments. Standard delivery takes 2-3 business days and costs $4.99. "
        "Express delivery takes 1 business day and costs $14.99. Zone A has no customs surcharge."},
    {"chunk_id": "zone_b_2024", "zone": "B", "effective_date": "2024-01-01", "is_current": False, "section": "Zone B rates (2024, superseded)", "text":
        "Zone B covers nearby international shipments. Standard delivery takes 5-7 business days and "
        "costs $9.99. Express delivery takes 2-3 business days and costs $24.99. A flat $6.00 customs "
        "surcharge applies to every Zone B order."},
    {"chunk_id": "zone_b_2026", "zone": "B", "effective_date": "2026-05-01", "is_current": True, "section": "Zone B rates (2026 update, current)", "text":
        "Effective May 2026, Zone B shipping rates have increased due to fuel surcharges. Standard "
        "delivery now costs $11.99. Express delivery remains $24.99. The customs surcharge increased "
        "to $8.00 per order."},
    {"chunk_id": "zone_c_2024", "zone": "C", "effective_date": "2024-01-01", "is_current": True, "section": "Zone C rates", "text":
        "Zone C covers remote international shipments. Standard delivery takes 10-14 business days and "
        "costs $19.99. Express delivery is not available for Zone C. A $15.00 customs surcharge applies."},
]

In [ ]:
# Step ("Enrich with semantic metadata"): prepend document/section context to the text that gets
# embedded, so the embedding carries more signal than the bare rate sentence alone.
for c in raw_chunks:
    c["text_for_embedding"] = f"Document: {DOC_TITLE}\nSection: {c['section']}\n\n{c['text']}"

ddl_create = f"""
CREATE OR REPLACE TABLE {SOURCE_TABLE} (
    chunk_id          STRING NOT NULL,
    zone              STRING,
    effective_date    DATE,
    is_current        BOOLEAN,
    section           STRING,
    text              STRING,
    text_for_embedding STRING
)
"""
sql_call(databricks_sdk_client=DATABRICKS_CLIENT, statement=ddl_create, warehouse_id=WAREHOUSE_ID)

values_sql = ",\n    ".join(
    "('{}', '{}', DATE('{}'), {}, '{}', '{}', '{}')".format(
        c["chunk_id"], c["zone"], c["effective_date"], str(c["is_current"]).upper(),
        c["section"].replace("'", "''"), c["text"].replace("'", "''"), c["text_for_embedding"].replace("'", "''"),
    )
    for c in raw_chunks
)
sql_call(
    databricks_sdk_client=DATABRICKS_CLIENT,
    statement=f"INSERT INTO {SOURCE_TABLE} VALUES\n    {values_sql}",
    warehouse_id=WAREHOUSE_ID,
)

In [ ]:
# Delta Sync Index requires Change Data Feed on the source table
sql_call(
    databricks_sdk_client=DATABRICKS_CLIENT,
    statement=f"ALTER TABLE {SOURCE_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)",
    warehouse_id=WAREHOUSE_ID,
)

print(f"Created and populated {SOURCE_TABLE} with {len(raw_chunks)} chunks "
      f"(zone B has both a 2024 [superseded] and a 2026 [current] version -- on purpose).")

### Create the endpoint and the Delta Sync Index for real

Idempotent: re-running this cell reuses the endpoint/index if they already exist instead of failing or recreating them. 
Endpoint and index creation each take a few minutes the first time.
We embed `text_for_embedding` (the enriched version) but still return the raw `text` column for display, 
so the enrichment improves retrieval without leaking the "Document: ... Section: ..."
header into the agent's final answer.


In [ ]:
from databricks.vector_search.client import VectorSearchClient

VS_ENDPOINT_NAME = "shipping_policy_endpoint"  # TODO: pick a name unique to your workspace
VS_INDEX_NAME = f"{VS_CATALOG}.{VS_SCHEMA}.shipping_policy_index"

vsc = VectorSearchClient()

if not vsc.endpoint_exists(VS_ENDPOINT_NAME):
    print(f"Creating endpoint {VS_ENDPOINT_NAME} (this can take a few minutes)...")
    vsc.create_endpoint_and_wait(name=VS_ENDPOINT_NAME, endpoint_type="STANDARD")
else:
    print(f"Endpoint {VS_ENDPOINT_NAME} already exists -- reusing it.")

if not vsc.index_exists(endpoint_name=VS_ENDPOINT_NAME, index_name=VS_INDEX_NAME):
    print(f"Creating index {VS_INDEX_NAME} (this can take a few minutes)...")
    vs_index = vsc.create_delta_sync_index_and_wait(
        endpoint_name=VS_ENDPOINT_NAME,
        index_name=VS_INDEX_NAME,
        source_table_name=SOURCE_TABLE,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="text_for_embedding",
        embedding_model_endpoint_name=EMBEDDING_ENDPOINT,
    )
else:
    print(f"Index {VS_INDEX_NAME} already exists -- reusing it.")
    vs_index = vsc.get_index(endpoint_name=VS_ENDPOINT_NAME, index_name=VS_INDEX_NAME)

# If you ever change shipping_policy_chunks after the index already exists (e.g. add a new
# zone_b_2027 row), re-trigger a sync so the index picks it up:
# vs_index.sync()

print(f"Index status: {vs_index.describe().get('status', {}).get('detailed_state')}")

### Query with and without a metadata filter on `effective_date`


In [ ]:
RESULT_COLUMNS = ["chunk_id", "zone", "effective_date", "is_current", "text"]

def print_vs_results(results, label):
    print(f"--- {label} ---")
    for row in results.get("result", {}).get("data_array", []):
        record = dict(zip(RESULT_COLUMNS + ["score"], row))
        print(f"  score={record['score']:.3f}  [{record['chunk_id']}, eff. {record['effective_date']}, current={record['is_current']}] {record['text'][:90]}...")


query = "What are delivery conditions to zone B ?"
CUTOFF_DATE = "2025-01-01"

no_filter_results = vs_index.similarity_search(
    query_text=query,
    columns=RESULT_COLUMNS,
    num_results=2,
)

filtered_results = vs_index.similarity_search(
    query_text=query,
    columns=RESULT_COLUMNS,
    filters={"effective_date >=": CUTOFF_DATE},
    num_results=2,
)

print_vs_results(no_filter_results, "WITHOUT a date filter")
print()
print_vs_results(filtered_results, f"WITH filter effective_date >= {CUTOFF_DATE}")

### Diagnosis

In practice, prefer filtering on a maintained `is_current` flag (set by your ingestion pipeline
when a newer version supersedes an old one) over a hardcoded cutoff date -- a fixed date only
works here because we know exactly when the rate changed. We now maintain that flag for real (see
the table above), and section 7 uses `is_current = true` for "current/now" questions instead of
date math -- date filters remain useful, but only for explicitly date-bounded historical questions
("what was the rate in 2024"), not for "what's the rate right now."


---
## 2 - Dynamic filter selection (column scope)

Section 1 hardcoded the filter (`effective_date >= CUTOFF_DATE`). In production you rarely know
the right filter ahead of time -- it depends on what the user actually asked. **Dynamic filter
selection** scopes the search down to the right metadata column(s) and value(s) *per query*,
following the two patterns from the
[AI Search retrieval quality guide](https://docs.databricks.com/aws/en/ai-search/retrieval-quality#dynamic-filter-selection):
a lightweight programmatic extractor, and an agent-based tool that lets the LLM generate the
filter itself.

### A - Programmatic filter extraction

Deterministic, no LLM call, easy to unit-test -- but only catches the patterns you wrote a rule
for. This mirrors the guide's `extract_filters()` example, adapted from `make`/`year` to our
`zone`/`effective_date`/`is_current` columns.


In [ ]:
import re

def extract_filters_programmatic(user_query):
    """Lightweight, deterministic filter extraction -- no LLM call, just pattern matching."""
    filters = {}

    zone_match = re.search(r"\bzone\s+([abc])\b", user_query, re.IGNORECASE)
    if zone_match:
        filters["zone"] = zone_match.group(1).upper()

    if re.search(r"\b(now|current|currently|today|latest)\b", user_query, re.IGNORECASE):
        filters["is_current"] = True
    else:
        year_match = re.search(r"\b(20\d{2})\b", user_query)
        if year_match:
            year = int(year_match.group(1))
            filters["effective_date >="] = f"{year}-01-01"
            filters["effective_date <"] = f"{year + 1}-01-01"

    return filters


test_queries = [
    "What does standard delivery to zone B cost right now?",
    "What was the zone A rate in 2024?",
    "Summarize shipping costs across all zones",
]
for q in test_queries:
    print(f"{q!r} -> {extract_filters_programmatic(q)}")

### B - Agent-based filtering with `VectorSearchRetrieverTool`

The guide's snippet imports `VectorSearchTool` from `databricks_ai_bridge.agents.tools.vector_search`,
but that module path doesn't exist in the published `databricks-ai-bridge` package (it raises
`ModuleNotFoundError`) -- the maintained, importable class is `databricks_langchain.VectorSearchRetrieverTool`.

It ships the native mechanism the guide describes: pass `dynamic_filter=True` and the tool inspects
the index's filterable columns and lets the LLM populate a structured `filters` argument itself at
call time. Each generated item is `{"key": "<column> <operator>", "value": ...}` -- `key` is the
*exact same string* you'd use as a dict key in the raw `filters` syntax from section 6, just wrapped
in a list of `{key, value}` objects instead of a dict.

**A real run of the previous version of this prompt surfaced a genuine logic bug, not just a
prompt-following slip**: for *"what does delivery to zone B cost right now?"*, asking the model to
filter `effective_date >= today` is backwards. If "today" is after a rate's effective date (which
it always is, for any rate currently in force), that filter excludes the very row that's actually
current. The earlier version only looked correct because the model under-shot the date (it used
January 1st instead of the literal today), which accidentally avoided the bug -- but the underlying
filter logic was wrong regardless of what the model did with it.

The fix is the same one section 6 already recommended and we now actually implement: filter on a
maintained `is_current` boolean for "now/current" questions, and reserve `effective_date` bounds
(both a lower **and** upper bound) for explicitly date-named historical questions. The
`tool_description` below reflects this with two few-shot examples, using exact key strings so the
model copies them verbatim rather than reconstructing operator syntax from prose.

In [ ]:
from databricks_langchain import VectorSearchRetrieverTool, ChatDatabricks

llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct")

TOOL_DESCRIPTION = """Search the shipping policy by zone and date.

Columns available for filtering:
- zone: one of "A", "B", "C".
- is_current: boolean. True means this is the rate currently in force for that zone.
- effective_date: a DATE in YYYY-MM-DD format, the date a rate started applying.

IMPORTANT -- filter keys must be copied EXACTLY as shown in the examples below: one space between
the column name and the operator, no trailing space, no extra characters.

Example 1 -- "current" / "now" / "currently" / "latest" rate questions. Use is_current, NOT a date
comparison -- filtering on today's date would incorrectly exclude a rate that became effective in
the past and is still in force:
  Question: "What does standard delivery to zone B cost right now?"
  Filters: [{"key": "zone", "value": "B"}, {"key": "is_current", "value": true}]

Example 2 -- "what was the rate in <year>" questions (always set BOTH bounds, so a rate later
superseded by a newer one is not returned for a question about the past):
  Question: "What was the zone A rate in 2024?"
  Filters: [{"key": "zone", "value": "A"}, {"key": "effective_date >=", "value": "2024-01-01"},
            {"key": "effective_date <", "value": "2025-01-01"}]
"""


In [ ]:

vector_search_tool = VectorSearchRetrieverTool(
    index_name=VS_INDEX_NAME,
    columns=RESULT_COLUMNS,
    num_results=2,
    tool_name="search_shipping_policy",
    tool_description=TOOL_DESCRIPTION,
    dynamic_filter=True,  # native mechanism: the LLM populates `filters` (List[FilterItem]) itself
)

llm_with_tool = llm.bind_tools([vector_search_tool])

for q in test_queries:
    response = llm_with_tool.invoke(q)
    tool_calls = getattr(response, "tool_calls", None) or []
    print(f"{q!r}")
    if not tool_calls:
        print(f"  -> no tool call; direct answer: {response.content[:160]}")
        continue
    for call in tool_calls:
        print(f"  -> filters selected by the agent: {call['args'].get('filters')}")
        try:
            tool_result = vector_search_tool.invoke(call["args"])
            print(f"  -> tool result: {str(tool_result)[:200]}")
        except Exception as e:
            print(f"  -> tool call failed: {e}")
            print(f"     (filters as generated: {call['args'].get('filters')} -- inspect operator")
            print(f"      and column type (DATE vs TIMESTAMP) on {SOURCE_TABLE} if this persists)")
    print()

### Programmatic vs agent-based

The agent-based tool above needed  prompt engineering (injecting the literal date, adding exact-key few-shot examples) 
just to stop failing on a 3-question test set -- and a different phrasing tomorrow could still trip it up in a new way. 
That fragility is the actual argument for keeping section A around, not a reason to drop it: cover the
small set of known, frequent patterns deterministically and for free (the guide's 90%+ search-space
reduction already comes from simple filters like these), and reach for the agent-based tool only as
a fallback for phrasings the regex genuinely can't anticipate -- accepting that it needs validation,
not blind trust, every time the prompt or the model changes.

### Cleanup

Uncomment and run when you're done experimenting -- the endpoint and index keep costing money
until deleted.


In [ ]:
vsc.delete_index(endpoint_name=VS_ENDPOINT_NAME, index_name=VS_INDEX_NAME)
vsc.delete_endpoint(VS_ENDPOINT_NAME)
print("Deleted index and endpoint.")